## Prompt Evals

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages = messages, stop_sequences=["```"])

    return json.loads(text)

In [5]:
dataset = generate_dataset()
print("Generated dataset:", dataset)

Generated dataset: [{'task': "Write a Python function that extracts the AWS region from an S3 bucket URL. For example, 's3://my-bucket.s3.us-west-2.amazonaws.com' should return 'us-west-2'."}, {'task': 'Create a JSON CloudFormation template snippet that defines an IAM policy allowing read-only access to all S3 buckets.'}, {'task': "Write a regular expression that matches valid AWS IAM role ARNs in the format 'arn:aws:iam::123456789012:role/RoleName'."}]


In [11]:
# Write the dataset to a JSON file
import os

os.makedirs("../datasets", exist_ok=True)

with open("../datasets/evaluation_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)